In [7]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
import plotly.graph_objects as go

#Frecuencia de muestreo del dataset
FS = 200.0
Ruta = Path("Archivos caídas\F06_SA01_R04.txt")

#columnas con los 9 canales contenidos en los dataset
COLUMNAS = [
    "ADXL_X", "ADXL_Y", "ADXL_Z",
    "ITG_X",  "ITG_Y",  "ITG_Z",
    "MMA_X",  "MMA_Y",  "MMA_Z"
]

# Factores de conversión de bits (ADC) a unidades físicas (g para acelerómetros, °/s para giroscopio)
FACTORES = {
    "ADXL": (2 * 16) / (2**13), # Rango +-16g, resolución 13 bits
    "ITG":  (2 * 2000) / (2**16), # Rango +-2000°/s, resolución 16 bits
    "MMA":  (2 * 8) / (2**14) # Rango +-8g, resolución 14 bits
}

In [8]:
def cargar_senal(ruta=Ruta, magnitud=True):
    #leemos el archivo de texto y lo convertimos a un dataframe de pandas
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)
    #la ultima columna contiene un caracter de punto y coma al final, lo eliminamos y convertimos a int
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)

    # Escalamos los datos crudos a unidades reales usandos los factores de conversión antes declarados
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor

    #calculamos la magnitud de cada sensor para que los datos no dependan de la posicion del sensor, 
    #si magnitud es True, se calcula la magnitud de cada sensor y se devuelve un dataframe con las magnitudes
    if magnitud:
        df = pd.DataFrame({
            prefijo: np.sqrt(df[f'{prefijo}_X']**2 + df[f'{prefijo}_Y']**2 + df[f'{prefijo}_Z']**2)
            for prefijo in FACTORES
        })
        
    return df

df_senal = cargar_senal()
print(f"Archivo cargado: {Ruta.name} | Filas: {len(df_senal)}")
print(df_senal.head())

Archivo cargado: F06_SA01_R04.txt | Filas: 3000
       ADXL        ITG       MMA
0  0.942734  54.145649  0.822916
1  0.954557  58.417309  0.829298
2  0.988798  60.687302  0.856301
3  1.019681  61.672092  0.888280
4  1.064867  61.690029  0.921408


In [9]:
nyq = FS / 2
GPASS = 1.0
GSTOP = 25.0

# --- RAMA 1: Filtro Pasa-Altas (Aísla el impacto rápido de la caída) ---
FRECHAZO_HPF = 6.0
FPASO_HPF = 11.0
orden_hpf, wn_hpf = signal.buttord(FPASO_HPF / nyq, FRECHAZO_HPF / nyq, GPASS, GSTOP)
sos_hpf = signal.butter(orden_hpf, wn_hpf, btype='highpass', output='sos')

# --- RAMA 2: Filtro Pasa-Bajas (Aísla el cambio lento de postura / gravedad) ---
FPASO_LPF = 0.5
FRECHAZO_LPF = 3.0
orden_lpf, wn_lpf = signal.buttord(FPASO_LPF / nyq, FRECHAZO_LPF / nyq, GPASS, GSTOP)
sos_lpf = signal.butter(orden_lpf, wn_lpf, btype='lowpass', output='sos')

def filtrar_ramas(df, columna='ADXL'):
    datos = df[columna].values
    # Fase cero (sosfiltfilt) en ambas ramas para mantener sincronía temporal exacta
    impacto = pd.Series(signal.sosfiltfilt(sos_hpf, datos), index=df.index)
    postura = pd.Series(signal.sosfiltfilt(sos_lpf, datos), index=df.index)
    return impacto, postura

senal_impacto, senal_postura = filtrar_ramas(df_senal, columna='ADXL')

print(f"HPF (Impacto) -> Orden: {orden_hpf} | Pico máximo: {senal_impacto.abs().max():.3f} g")
print(f"LPF (Postura) -> Orden: {orden_lpf} | Rango postura: [{senal_postura.min():.3f}, {senal_postura.max():.3f}] g")

HPF (Impacto) -> Orden: 6 | Pico máximo: 0.287 g
LPF (Postura) -> Orden: 2 | Rango postura: [0.869, 1.552] g


In [10]:
from plotly.subplots import make_subplots

def graficar_bode(sos, orden, wn, tipo_texto, f_especificaciones):
    # Calculamos la respuesta en frecuencia del filtro digital
    w, h = signal.sosfreqz(sos, worN=2048, fs=FS)
    
    # Magnitud en dB (un paso vs. ida y vuelta con filtfilt)
    mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    mag_eff_db = 2 * mag_db
    # Fase continua desenrollada en grados
    fase_deg = np.unwrap(np.angle(h)) * (180 / np.pi)
    fc_hz = wn * (FS / 2)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=(
            f"Diagrama de Bode: {tipo_texto} (Orden {orden}, Wn = {fc_hz:.2f} Hz)",
            "Fase"
        )
    )

    # Subgráfica 1: Magnitud
    fig.add_trace(go.Scatter(x=w, y=mag_db, name="Un paso (freqz)", line=dict(color="#1f77b4")), row=1, col=1)
    fig.add_trace(go.Scatter(x=w, y=mag_eff_db, name="Efectiva (filtfilt)", line=dict(color="red", dash="dash")), row=1, col=1)
    
    # Líneas verticales de referencia (Wn, fp y fs)
    fig.add_vline(x=fc_hz, line_dash="dash", line_color="gray", annotation_text=f"Wn={fc_hz:.2f} Hz", row=1, col=1)
    for etiqueta, freq in f_especificaciones.items():
        fig.add_vline(x=freq, line_dash="dot", line_color="gray", annotation_text=f"{etiqueta}={freq} Hz", row=1, col=1)

    # Subgráfica 2: Fase
    fig.add_trace(go.Scatter(x=w, y=fase_deg, name="Fase un paso", line=dict(color="#1f77b4"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=w, y=np.zeros_like(w), name="Fase efectiva (0°)", line=dict(color="red", dash="dash"), showlegend=False), row=2, col=1)

    fig.update_xaxes(type="log", range=[np.log10(0.1), np.log10(FS / 2)], title_text="Frecuencia (Hz)", row=2, col=1)
    fig.update_yaxes(range=[-100, 5], title_text="Magnitud (dB)", row=1, col=1)
    fig.update_yaxes(title_text="Fase (°)", row=2, col=1)
    fig.update_layout(height=480, legend=dict(orientation="h", y=1.15))
    fig.show()

def graficar_ramas_tiempo(df=df_senal, impacto=senal_impacto, postura=senal_postura, ruta=Ruta, columna='ADXL'):
    clase = "ADL" if ruta.name.startswith("D") else "Caída"
    t = np.arange(len(df)) / FS

    # 3 subgráficas compartiendo el eje de tiempo para comparar ambas ramas
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=(
            f"1. Señal Original ({columna})",
            f"2. Rama HPF — Impacto (Pico máx: {impacto.abs().max():.3f} g)",
            f"3. Rama LPF — Postura / Tendencia lenta (Rango: {postura.min():.2f} a {postura.max():.2f} g)"
        )
    )

    fig.add_trace(go.Scatter(x=t, y=df[columna], name="Original", line=dict(color="gray", width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=impacto, name="HPF (Impacto)", line=dict(color="#1f77b4", width=1.2)), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=postura, name="LPF (Postura)", line=dict(color="#2ca02c", width=1.5)), row=3, col=1)

    fig.update_xaxes(title_text="Tiempo (s)", row=3, col=1)
    fig.update_yaxes(title_text="g", row=1, col=1)
    fig.update_yaxes(title_text="g", row=2, col=1)
    fig.update_yaxes(title_text="g", row=3, col=1)
    fig.update_layout(
        title_text=f"Procesamiento en Paralelo: {ruta.stem} ({clase})",
        height=650,
        hovermode="x unified",
        showlegend=False
    )
    fig.show()

# Ejecutamos los diagramas de Bode de ambos filtros y la comparación temporal
graficar_bode(sos_hpf, orden_hpf, wn_hpf, "Butterworth Highpass (Impacto)", {"fs": FRECHAZO_HPF, "fp": FPASO_HPF})
graficar_bode(sos_lpf, orden_lpf, wn_lpf, "Butterworth Lowpass (Postura)", {"fp": FPASO_LPF, "fs": FRECHAZO_LPF})
graficar_ramas_tiempo()

In [11]:
# Umbrales calibrados para la arquitectura de doble rama (HPF + LPF)
UMBRAL_IMPACTO_FUERTE = 1.75   # Pico HPF (g) para caídas bruscas
UMBRAL_IMPACTO_SUAVE  = 0.85   # Pico HPF (g) mínimo si va seguido de quietud absoluta en el suelo
UMBRAL_QUIETUD_LPF    = 0.08   # Desviación estándar máxima en LPF (g) tras el impacto (indica inmovilidad)

def detectar_caida_paralela(impacto, postura,
                            u_fuerte=UMBRAL_IMPACTO_FUERTE,
                            u_suave=UMBRAL_IMPACTO_SUAVE,
                            u_quietud=UMBRAL_QUIETUD_LPF):
    # 1. Rama HPF: magnitud del pico de impacto e índice temporal donde ocurre
    abs_impacto = impacto.abs()
    pico = float(abs_impacto.max())
    idx_pico = int(abs_impacto.idxmax())
    
    # 2. Rama LPF: analizamos la estabilidad de la postura después del impacto
    inicio_post = min(idx_pico + int(0.5 * FS), len(postura) - 1)
    fin_post    = min(idx_pico + int(2.5 * FS), len(postura))
    
    if fin_post - inicio_post >= int(0.5 * FS):
        variacion_post = float(postura.iloc[inicio_post:fin_post].std())
    else:
        variacion_post = float(postura.iloc[-int(FS):].std())
        
    esta_quieto_despues = variacion_post < u_quietud
    
    # 3. Regla de decisión combinada (fusión de ambas ramas)
    es_caida = (pico > u_fuerte) or (pico > u_suave and esta_quieto_despues)
    etiqueta = "Caída" if es_caida else "ADL"
    
    return es_caida, etiqueta, pico, variacion_post

# Obtenemos la etiqueta real a partir de la nomenclatura del dataset SisFall ('D' = ADL, 'F' = Caída)
clase_real = "ADL" if Ruta.name.startswith("D") else "Caída"

# Evaluamos ambas ramas filtradas con el clasificador paralelo
es_caida, etiqueta, pico, var_post = detectar_caida_paralela(senal_impacto, senal_postura)

# Imprimimos el resumen comparando la predicción del sistema contra la clase real
print(f"Archivo:        {Ruta.name}")
print(f"Predicho:       {etiqueta}")
print(f"Real:           {clase_real}")
print(f"Detalle ramas:  Pico HPF = {pico:.4f} g | Oscilación post-impacto LPF = {var_post:.4f} g")

Archivo:        F06_SA01_R04.txt
Predicho:       ADL
Real:           Caída
Detalle ramas:  Pico HPF = 0.2867 g | Oscilación post-impacto LPF = 0.0428 g


In [12]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd

# Aseguramos que la carpeta apunte al directorio padre de tu ruta de ejemplo
CARPETA = Ruta.parent 

def extraer_caracteristicas_dataset(carpeta=CARPETA, columna='ADXL'):
    """
    Recorre todos los archivos .txt de la carpeta, aplica las dos ramas en paralelo
    (HPF impacto + LPF postura) y extrae el pico máximo y la oscilación post-impacto.
    """
    registros = []
    for ruta_txt in sorted(glob.glob(str(Path(carpeta) / "*.txt"))):
        ruta_obj = Path(ruta_txt)
        nombre = ruta_obj.stem
        
        # Ignoramos archivos auxiliares que no empiecen por D (ADL) o F (Caída)
        if not (nombre.startswith("D") or nombre.startswith("F")):
            continue
            
        df = cargar_senal(ruta=ruta_obj, magnitud=True)
        impacto, postura = filtrar_ramas(df, columna=columna)
        
        # 1. Característica Rama HPF: pico máximo absoluto e índice temporal
        abs_imp = impacto.abs()
        pico = float(abs_imp.max())
        idx_pico = int(abs_imp.idxmax())
        
        # 2. Característica Rama LPF: desviación estándar en la ventana post-impacto (0.5 s a 2.5 s)
        inicio_post = min(idx_pico + int(0.5 * FS), len(postura) - 1)
        fin_post    = min(idx_pico + int(2.5 * FS), len(postura))
        if fin_post - inicio_post >= int(0.5 * FS):
            var_post = float(postura.iloc[inicio_post:fin_post].std())
        else:
            var_post = float(postura.iloc[-int(FS):].std())
            
        clase_bin = 1 if nombre.startswith("F") else 0
        registros.append({
            'Archivo': nombre,
            'Clase': 'Caída' if clase_bin == 1 else 'ADL',
            'y': clase_bin,
            'Pico_HPF_g': pico,
            'Var_Post_LPF_g': var_post
        })
        
    return pd.DataFrame(registros)


def evaluar_loocv_paralelo(df_datos, nombre_escenario=""):
    """
    Ejecuta Leave-One-Out Cross-Validation optimizando la Exactitud Balanceada
    para el clasificador combinado (Impacto fuerte HPF O impacto moderado + quietud LPF).
    """
    df = df_datos.reset_index(drop=True)
    candidatos_fuerte = np.linspace(df['Pico_HPF_g'].min(), df['Pico_HPF_g'].max(), 200)
    
    umbrales_fold = []
    predicciones = []
    
    for i in range(len(df)):
        train = df.drop(index=i)
        y_tr = train['y'].values
        picos_tr = train['Pico_HPF_g'].values
        quieto_tr = (train['Var_Post_LPF_g'].values < UMBRAL_QUIETUD_LPF) & (picos_tr > UMBRAL_IMPACTO_SUAVE)
        
        # Buscamos el umbral óptimo de impacto fuerte en el conjunto de entrenamiento (N-1)
        def score_fn(u):
            pred_tr = (picos_tr >= u) | quieto_tr
            sens = pred_tr[y_tr == 1].mean()
            espec = (~pred_tr[y_tr == 0]).mean()
            return 0.5 * (sens + espec)
            
        mejor_u = max(candidatos_fuerte, key=score_fn)
        umbrales_fold.append(mejor_u)
        
        # Evaluamos el archivo excluido (test) con la regla aprendida
        pico_test = df.loc[i, 'Pico_HPF_g']
        quieto_test = (df.loc[i, 'Var_Post_LPF_g'] < UMBRAL_QUIETUD_LPF) and (pico_test > UMBRAL_IMPACTO_SUAVE)
        predicciones.append(int((pico_test >= mejor_u) or quieto_test))
        
    df['Umbral_LOO_g'] = umbrales_fold
    df['Pred_y'] = predicciones
    df['Acierto'] = df['y'] == df['Pred_y']
    
    caidas = df[df['y'] == 1]
    adls   = df[df['y'] == 0]
    
    sensibilidad  = caidas['Acierto'].mean() * 100
    especificidad = adls['Acierto'].mean() * 100
    acc_global    = df['Acierto'].mean() * 100
    acc_balance   = (sensibilidad + especificidad) / 2
    
    resumen = {
        'Escenario': nombre_escenario,
        'Total Archivos': len(df),
        'Umbral HPF Prom. (g)': round(np.mean(umbrales_fold), 4),
        'Sensibilidad (%)': round(sensibilidad, 2),
        'Especificidad (%)': round(especificidad, 2),
        'Exactitud Balanceada (%)': round(acc_balance, 2),
        'Exactitud Global (%)': round(acc_global, 2),
        'Caídas Detectadas': f"{caidas['Acierto'].sum()}/{len(caidas)}"
    }
    
    return df, resumen


# ==============================================================================
# EJECUCIÓN DE LA EVALUACIÓN LOOCV (DOBLE RAMA HPF + LPF)
# ==============================================================================

# 1. Extraemos características de ambas ramas para toda la carpeta
df_completo = extraer_caracteristicas_dataset(CARPETA, columna='ADXL')

# Verificamos los valores del archivo anómalo D04_SA03_R01
fila_anomala = df_completo[df_completo['Archivo'] == 'D04_SA03_R01']
if not fila_anomala.empty:
    p_anom = fila_anomala['Pico_HPF_g'].values[0]
    v_anom = fila_anomala['Var_Post_LPF_g'].values[0]
    print(f"🔎 D04_SA03_R01 -> Pico HPF: {p_anom:.4f} g | Oscilación Post-LPF: {v_anom:.4f} g\n")

# 2. Escenario A: Dataset completo (incluyendo D04_SA03_R01)
df_res_con, metricas_con = evaluar_loocv_paralelo(df_completo, "1. Con D04_SA03_R01 (Doble Rama)")

# 3. Escenario B: Dataset depurado (descartando D04_SA03_R01)
df_sin_anomalo = df_completo[df_completo['Archivo'] != 'D04_SA03_R01'].copy()
df_res_sin, metricas_sin = evaluar_loocv_paralelo(df_sin_anomalo, "2. Sin D04_SA03_R01 (Doble Rama)")

# 4. Tabla comparativa de resultados
df_comparacion = pd.DataFrame([metricas_con, metricas_sin]).set_index('Escenario')
display(df_comparacion)

# 5. Identificar qué caídas leves se rescatan al eliminar la anomalía
caidas_ganadas = set(df_res_sin[(df_res_sin['y'] == 1) & (df_res_sin['Acierto'])]['Archivo']) - \
                 set(df_res_con[(df_res_con['y'] == 1) & (df_res_con['Acierto'])]['Archivo'])

print(f"\n✔ Caídas leves rescatadas al descartar D04_SA03_R01 ({len(caidas_ganadas)}):")
if caidas_ganadas:
    display(df_res_sin[df_res_sin['Archivo'].isin(caidas_ganadas)][['Archivo', 'Pico_HPF_g', 'Var_Post_LPF_g', 'Umbral_LOO_g']])
else:
    print("Ninguna caída adicional superó el nuevo umbral.")

🔎 D04_SA03_R01 -> Pico HPF: 1.7222 g | Oscilación Post-LPF: 0.0537 g



,Total Archivos,Umbral HPF Prom. (g),Sensibilidad (%),Especificidad (%),Exactitud Balanceada (%),Exactitud Global (%),Caídas Detectadas
Escenario,,,,,,,
1. Con D04_SA03_R01 (Doble Rama),68,0.4222,96.67,12.50,54.58,86.76,58/60
2. Sin D04_SA03_R01 (Doble Rama),67,0.4221,96.67,14.29,55.48,88.06,58/60



✔ Caídas leves rescatadas al descartar D04_SA03_R01 (0):
Ninguna caída adicional superó el nuevo umbral.
